# 10. 피벗 테이블과 크로스탭

## 학습 목표
- `pivot_table`로 2차원 집계표를 만들 수 있다
- 행/열/값/집계함수 옵션을 이해한다
- `crosstab`으로 빈도 분석을 할 수 있다
- `margins`로 합계를 추가할 수 있다

---

## 왜 배워야 하나요?

엑셀의 "피벗 테이블"이 Pandas에도 그대로 있습니다.
**행 × 열 × 값** 형태로 데이터를 빠르게 요약할 수 있어 **보고서 작성**에 매우 유용합니다.

`groupby` + `unstack` 조합을 더 편하게 쓸 수 있는 문법이라고 생각하면 됩니다.


In [1]:
# 아래에 코드를 작성하세요
import pandas as pd

In [2]:
q1 = pd.read_csv('./data/orders_q1.csv')
q2 = pd.read_csv('./data/orders_q2.csv')
q3 = pd.read_csv('./data/orders_q3.csv')
q4 = pd.read_csv('./data/orders_q4.csv')

In [8]:
q1.head()

,주문번호,주문일자,고객ID,상품코드,수량
0,ORD00001,2024-01-02,C019,P001,3
1,ORD00002,2024-01-03,C028,P013,3
2,ORD00003,2024-01-04,C026,P002,3
3,ORD00004,2024-01-05,C037,P004,2
4,ORD00005,2024-01-06,C026,P014,2


In [4]:
q2.head()

,주문번호,주문일자,고객ID,상품코드,수량
0,ORD00151,2024-04-01,C029,P012,2
1,ORD00152,2024-04-01,C038,P012,2
2,ORD00153,2024-04-02,C033,P005,1
3,ORD00154,2024-04-02,C037,P007,2
4,ORD00155,2024-04-03,C027,P003,2


In [11]:
orders = pd.concat( [ q1, q2, q3, q4 ] , ignore_index= True )

In [12]:
orders.head()

,주문번호,주문일자,고객ID,상품코드,수량
0,ORD00001,2024-01-02,C019,P001,3
1,ORD00002,2024-01-03,C028,P013,3
2,ORD00003,2024-01-04,C026,P002,3
3,ORD00004,2024-01-05,C037,P004,2
4,ORD00005,2024-01-06,C026,P014,2


In [14]:
products = pd.read_csv('./data/products.csv')

In [16]:
customers = pd.read_csv('./data/customers.csv')

In [21]:
df = pd.merge( orders , products , on= '상품코드'  )

In [24]:
df = pd.merge( df , customers, on = '고객ID' )

In [25]:
df.head(3)

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가,고객명,성별,연령,지역,가입일,회원등급
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000,엄정화,여,54,대전,2022-04-01,골드
1,ORD00002,2024-01-03,C028,P013,3,이어폰,음향,55000,김유정,여,53,서울,2022-05-16,실버
2,ORD00003,2024-01-04,C026,P002,3,무선마우스,주변기기,35000,김민수,여,27,경기,2022-05-06,실버


In [28]:
df['매출'] = df['수량'] * df['단가']

In [29]:
df.head(3)

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가,고객명,성별,연령,지역,가입일,회원등급,매출
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000,엄정화,여,54,대전,2022-04-01,골드,4050000
1,ORD00002,2024-01-03,C028,P013,3,이어폰,음향,55000,김유정,여,53,서울,2022-05-16,실버,165000
2,ORD00003,2024-01-04,C026,P002,3,무선마우스,주변기기,35000,김민수,여,27,경기,2022-05-06,실버,105000


In [31]:
df.to_csv('./data/all.csv', index=False)

---
## 1. `pivot_table` 기본

```python
df.pivot_table(
    index="행 기준 컬럼",
    columns="열 기준 컬럼",
    values="집계 대상 컬럼",
    aggfunc="합계/평균/count 등"
)
```


In [33]:
# 아래에 코드를 작성하세요
# 지역을 인덱스로 놓고, 컬럼을 카테고리로 놓자. 매출을 데이터부분에 넣자. => 매출의 합으로 넣자.
df.pivot_table(index='지역', columns='카테고리', values='매출', aggfunc='sum')


카테고리,액세서리,웨어러블,음향,저장장치,주변기기,컴퓨터
지역,,,,,,
경기,1350000.0,8320000.0,4560000.0,3515000.0,10588000.0,58840000.0
광주,372000.0,5120000.0,745000.0,1710000.0,1391000.0,6080000.0
대구,320000.0,2240000.0,1465000.0,1425000.0,3164000.0,18690000.0
대전,592000.0,640000.0,1485000.0,1045000.0,2477000.0,40050000.0
부산,24000.0,NaN,480000.0,95000.0,788000.0,2250000.0
서울,1608000.0,8960000.0,8465000.0,3135000.0,9323000.0,68670000.0
인천,112000.0,640000.0,765000.0,1140000.0,1689000.0,15100000.0


In [ ]:
# 아래에 코드를 작성하세요
# 인덱스는  카테고리, 데이터는 매출,  컬럼은 매출의 총합, 평균, 주문갯수
df.pivot_table(index='카테고리', values='매출', aggfunc= ['sum', 'mean', 'count'] )


---
## 2. `margins` - 합계 행/열 추가


In [ ]:
# 아래에 코드를 작성하세요


---
## 3. `crosstab` - 빈도 분석

`pivot_table`의 특수한 형태로, **빈도(count)** 를 빠르게 구합니다.


In [ ]:
# 아래에 코드를 작성하세요


In [ ]:
# 아래에 코드를 작성하세요


---
## 실습문제

> 공통 준비:
> ```python
> import glob
> orders = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("../data/orders_q*.csv"))], ignore_index=True)
> products = pd.read_csv("../data/products.csv")
> customers = pd.read_csv("../data/customers.csv")
> df = orders.merge(products, on="상품코드").merge(customers[["고객ID", "성별", "지역"]], on="고객ID")
> df["매출"] = df["수량"] * df["단가"]
> ```

### 문제 1. 지역 × 카테고리 매출 피벗

`지역`(행) × `카테고리`(열)로 매출 합계를 구하고, 합계 행/열까지 추가하세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 2. 성별 × 카테고리 주문 건수

성별과 카테고리별 **주문 건수**를 `crosstab`으로 구하세요. 합계도 포함하세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 3. 카테고리별 매출 합계와 평균

카테고리별로 매출의 **합계와 평균**을 한번에 구하는 피벗 테이블을 만드세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 4. 지역 × 성별 비율(%)

`지역`과 `성별`의 **행 비율(%)** 크로스탭을 구하세요 (각 지역 내에서 남/여 비율).


In [ ]:
# 아래에 코드를 작성하세요
